<a href="https://colab.research.google.com/github/Soumita2026/Soumita_Deb/blob/master/Copy_of_updated_code.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Setup and Installation
This cell installs the necessary Python libraries for building a Retrieval-Augmented Generation (RAG) pipeline using LlamaIndex. It includes components for language models (LLMs), embeddings, retrievers (BM25), document readers, and asynchronous operations.

In [5]:
!pip -q install -U \
  llama-index \
  llama-index-llms-openai \
  llama-index-embeddings-openai \
  llama-index-embeddings-huggingface \
  llama-index-llms-llama-cpp \
  llama-index-retrievers-bm25 \
  llama-index-readers-file \
  nest-asyncio

In [6]:
import torch

# Check if GPU is available
if torch.cuda.is_available():
    print("GPU is available, using GPU.")
    device = "cuda"
else:
    print("GPU not available, using CPU.")
    device = "cpu"

!pip -q install -U sentence-transformers

GPU not available, using CPU.


In [7]:
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

Using `BAAI/bge-small-en-v1.5` as the local embedding model, which is a good balance between performance and size.

In [11]:
from llama_index.core import Settings
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
Settings.embed_model = HuggingFaceEmbedding(
    model_name="BAAI/bge-small-en-v1.5",
    device=device
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [14]:
# Re-running the installation cell with the added package
# and other relevant cells after applying fixes
!pip -q install -U \
  llama-index \
  llama-index-llms-openai \
  llama-index-embeddings-openai \
  llama-index-embeddings-huggingface \
  llama-index-retrievers-bm25 \
  llama-index-readers-file \
  nest-asyncio

# Re-running the import for HuggingFaceEmbedding
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

# Re-running the LlamaIndex settings configuration (now without OpenAI embedding)
from llama_index.llms.openai import OpenAI
from llama_index.core.node_parser import SentenceSplitter
from llama_index.core import Settings

Settings.llm = OpenAI(model="gpt-4o-mini", temperature=0)
Settings.node_parser = SentenceSplitter(chunk_size=256, chunk_overlap=40)

# Re-running the setting of the embedding model to HuggingFaceEmbedding
import torch
if torch.cuda.is_available():
    device = "cuda"
else:
    device = "cpu"
Settings.embed_model = HuggingFaceEmbedding(
    model_name="BAAI/bge-small-en-v1.5",
    device=device
)

# Now, attempting to create the vector store index with the local embedding model
from llama_index.core import VectorStoreIndex, SimpleDirectoryReader
import os
import textwrap

# Ensure the directory exists and is populated with files within this cell
os.makedirs("tech_kb", exist_ok=True)

docs = {
    "rag_basics.txt": """Retrieval-Augmented Generation, or RAG, combines retrieval with language generation.\nA typical RAG pipeline retrieves relevant chunks and uses them as context for the LLM.\nRAG reduces hallucination by grounding answers in external knowledge.\nHowever, vector-only retrieval may miss results when exact keywords matter or when the query has multiple parts.""",

    "hybrid_search.txt": """Hybrid retrieval combines dense vector retrieval and sparse BM25 retrieval.\nDense retrieval captures semantic similarity between query and text.\nBM25 captures keyword overlap and term importance.\nReciprocal Rank Fusion combines ranked lists from multiple retrievers to improve retrieval quality.""",

    "reranking_note.txt": """In production systems, a second-stage reranker may be used after retrieval.\nIn this lab, we use reciprocal rank fusion as the ranking strategy, so no separate reranker API is required.""",

    "query_decomposition.txt": """Query decomposition breaks one complex user question into smaller sub-queries.\nThis improves recall because each sub-query can retrieve useful evidence.\nFusion-based retrieval systems often combine query decomposition with multi-retriever search.""",

    "citations.txt": """Citations show which source documents support the generated answer.\nThey improve trust, transparency, and verification in a RAG system.\nLlamaIndex can return source nodes along with the final response."""
}

for filename, content in docs.items():
    with open(os.path.join("tech_kb", filename), "w", encoding="utf-8") as f:
        f.write(textwrap.dedent(content).strip())

# Load documents within this cell
documents = SimpleDirectoryReader("tech_kb").load_data()

vector_index = VectorStoreIndex.from_documents(documents)
print("Vector index ready with local embedding model")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Vector index ready with local embedding model


### Asynchronous Operations Fix
`nest_asyncio` is imported and `nest_asyncio.apply()` is called to allow nested execution of asyncio event loops. This is often necessary in environments like Jupyter or Colab where an event loop might already be running, preventing other asynchronous operations.

In [15]:
import nest_asyncio
nest_asyncio.apply()

### API Key Configuration
This cell imports the `os` module to manage environment variables and `userdata` from `google.colab` to securely retrieve API keys. It then sets the OpenAI API key as an environment variable, which is used by LlamaIndex to authenticate with OpenAI services.

In [16]:
import os
from google.colab import userdata

os.environ["OPENAI_API_KEY"] = userdata.get("APIKEY")

print("OpenAI key loaded:", bool(os.environ.get("OPENAI_API_key")))


OpenAI key loaded: False


### Create Knowledge Base Directory
`os.makedirs` is used to create a directory named `tech_kb`. The `exist_ok=True` argument ensures that if the directory already exists, no error is raised.

In [17]:
os.makedirs("tech_kb", exist_ok=True)

### Define Knowledge Base Content
This dictionary, `docs`, stores the content for our knowledge base. Each key is a filename, and its value is a string containing textual information about RAG, hybrid search, reranking, query decomposition, and citations.

In [18]:
docs = {
    "rag_basics.txt": """Retrieval-Augmented Generation, or RAG, combines retrieval with language generation.\nA typical RAG pipeline retrieves relevant chunks and uses them as context for the LLM.\nRAG reduces hallucination by grounding answers in external knowledge.\nHowever, vector-only retrieval may miss results when exact keywords matter or when the query has multiple parts.""",

    "hybrid_search.txt": """Hybrid retrieval combines dense vector retrieval and sparse BM25 retrieval.\nDense retrieval captures semantic similarity between query and text.\nBM25 captures keyword overlap and term importance.\nReciprocal Rank Fusion combines ranked lists from multiple retrievers to improve retrieval quality.""",

    "reranking_note.txt": """In production systems, a second-stage reranker may be used after retrieval.\nIn this lab, we use reciprocal rank fusion as the ranking strategy, so no separate reranker API is required.""",

    "query_decomposition.txt": """Query decomposition breaks one complex user question into smaller sub-queries.\nThis improves recall because each sub-query can retrieve useful evidence.\nFusion-based retrieval systems often combine query decomposition with multi-retriever search.""",

    "citations.txt": """Citations show which source documents support the generated answer.\nThey improve trust, transparency, and verification in a RAG system.\nLlamaIndex can return source nodes along with the final response."""
}

### Write Content to Files
This cell iterates through the `docs` dictionary. For each entry, it uses `textwrap.dedent` to remove common leading whitespace from multiline strings and then writes the content to a corresponding text file inside the `tech_kb` directory. This simulates having multiple knowledge base documents.

In [19]:
import textwrap
for filename, content in docs.items():
    with open(os.path.join("tech_kb", filename), "w", encoding="utf-8") as f:
        f.write(textwrap.dedent(content).strip())

### Load Documents
`SimpleDirectoryReader` from LlamaIndex is used to load all text documents from the `tech_kb` directory. These documents will form the basis of our RAG system's knowledge.

In [20]:
from llama_index.core import SimpleDirectoryReader
documents = SimpleDirectoryReader("tech_kb").load_data()
print("Documents loaded:", len(documents))

Documents loaded: 5


### Configure LlamaIndex Settings
This cell imports and configures core components of LlamaIndex:
- `OpenAI`: Sets the Language Model (LLM) to 'gpt-4o-mini' with a temperature of 0 for deterministic responses.
- `OpenAIEmbedding`: Sets the embedding model to 'text-embedding-3-small' for generating numerical representations of text.
- `SentenceSplitter`: Configures the node parser to split documents into chunks of 256 tokens with an overlap of 40 tokens. This helps in processing documents into manageable units for retrieval.

In [21]:
# from llama_index.llms.openai import OpenAI # Commented out as we are using a local LLM
# from llama_index.embeddings.openai import OpenAIEmbedding # Commented out as we are using HuggingFaceEmbedding
from llama_index.core.node_parser import SentenceSplitter
from llama_index.core import Settings

# Settings.llm = OpenAI(model="gpt-4o-mini", temperature=0) # Commented out as we are using a local LLM
# Settings.embed_model = OpenAIEmbedding(model="text-embedding-3-small") # Commented out as we are using HuggingFaceEmbedding
Settings.node_parser = SentenceSplitter(chunk_size=256, chunk_overlap=40)

### Create Vector Store Index
`VectorStoreIndex.from_documents` creates an index from the loaded documents. This index uses the configured embedding model to convert document chunks into vectors, which allows for semantic similarity search.

In [22]:
from llama_index.core import VectorStoreIndex

vector_index = VectorStoreIndex.from_documents(documents)
print("Vector index ready")

Vector index ready


### Initialize Vector Retriever
This cell converts the `vector_index` into a `vector_retriever`. This retriever is responsible for fetching the top `similarity_top_k` (4 in this case) most semantically similar document chunks based on a query.

In [23]:
vector_retriever = vector_index.as_retriever(similarity_top_k=4)

### Initialize BM25 Retriever
This cell sets up a `BM25Retriever`, which is a sparse retrieval method that focuses on keyword matching and term frequency. It first gets nodes (chunks) from the documents using the configured `SentenceSplitter` and then initializes the BM25 retriever with these nodes, also configured to return the top 4 results.

In [24]:
# create the BM25 Retiever. I don't want my search to be based solely on context but I want it based on keyword matching
from llama_index.retrievers.bm25 import BM25Retriever
from llama_index.core import Settings

nodes = Settings.node_parser.get_nodes_from_documents(documents)
bm25_retriever = BM25Retriever.from_defaults(nodes=nodes, similarity_top_k=4)

DEBUG:bm25s:Building index from IDs objects


### Create Query Fusion Retriever
`QueryFusionRetriever` combines the results from multiple retrievers (in this case, the `vector_retriever` and `bm25_retriever`). It uses `reciprocal_rerank` mode to combine and re-rank the results, aiming to improve retrieval quality by leveraging both semantic and keyword-based search. `num_queries` is set to 4 to generate additional sub-queries for broader retrieval.

In [25]:
# create a Query fusion retriver to combine the results from both the vctor and BM25
from llama_index.core.retrievers import QueryFusionRetriever

fusion_retriever = QueryFusionRetriever(
    [vector_retriever, bm25_retriever],
    similarity_top_k=6,
    num_queries=4,
    mode="reciprocal_rerank",
    use_async=True,
    verbose=True,
)

print("Hybrid fusion retriever created")

Hybrid fusion retriever created


### Configure Response Synthesizer
`get_response_synthesizer` creates an object responsible for generating the final answer from the retrieved document chunks. The `response_mode='compact'` setting instructs it to use a compact approach for synthesizing the response, summarizing information efficiently.

In [26]:
from llama_index.core.response_synthesizers import get_response_synthesizer

response_synthesizer = get_response_synthesizer(response_mode="compact")

### Create Retriever Query Engine
This cell initializes the `RetrieverQueryEngine`, which is the core component of our RAG pipeline. It takes the `fusion_retriever` to retrieve relevant context and the `response_synthesizer` to generate an answer based on that context. This engine orchestrates the entire query process.

In [27]:
from llama_index.core.query_engine import RetrieverQueryEngine

query_engine = RetrieverQueryEngine(
    retriever=fusion_retriever,
    response_synthesizer=response_synthesizer,
)

print("Advanced RAG pipeline ready")

Advanced RAG pipeline ready


### Define the Query
This markdown cell simply defines the `query` string that will be passed to the RAG pipeline. The query asks for an explanation of several advanced RAG concepts and the importance of citations.

In [28]:
query = """
Explain how query decomposition, hybrid retrieval, and reciprocal rank fusion
work together in an advanced RAG pipeline. Also mention why citations matter.
"""

### Execute the Query
This cell executes the defined `query` using the `query_engine`. The `query_engine` will perform retrieval, synthesis, and generate a response based on the knowledge base.

In [ ]:
response = query_engine.query(query)

### Switch to Local LLM (LlamaCPP)

First, we need to install `llama-cpp-python`, which allows us to run local LLMs in GGUF format.

In [29]:
!pip -q install llama-cpp-python --force-reinstall --no-deps

Next, we'll download a local GGUF model. I'll use `NousResearch/Nous-Hermes-2-Mistral-7B-DPO-GGUF` as an example. This model offers a good balance of performance and size for local execution.

In [30]:
from huggingface_hub import hf_hub_download
import os

model_name = "NousResearch/Nous-Hermes-2-Mistral-7B-DPO-GGUF"
model_file = "Nous-Hermes-2-Mistral-7B-DPO.Q4_K_M.gguf" # Corrected filename

model_path = hf_hub_download(repo_id=model_name, filename=model_file)

print(f"Downloaded model to: {model_path}")

Nous-Hermes-2-Mistral-7B-DPO.Q4_K_M.gguf:   0%|          | 0.00/4.37G [00:00<?, ?B/s]

Downloaded model to: /root/.cache/huggingface/hub/models--NousResearch--Nous-Hermes-2-Mistral-7B-DPO-GGUF/snapshots/eb85cf06e8663157611e8ee472e61b43f50ee49f/Nous-Hermes-2-Mistral-7B-DPO.Q4_K_M.gguf


Now, we will configure LlamaIndex to use this local LLM by importing `LlamaCPP` and setting `Settings.llm`.

In [31]:
from llama_index.llms.llama_cpp import LlamaCPP

llm = LlamaCPP(
    model_path=model_path,
    temperature=0.1,
    max_new_tokens=256,
    context_window=3900,
    generate_kwargs={},
    model_kwargs={"n_gpu_layers": -1}, # Uncomment this line to use GPU if available
    verbose=True,
)

Settings.llm = llm

print("LlamaIndex LLM configured to use local LlamaCPP model.")

llama_model_loader: loaded meta data with 22 key-value pairs and 291 tensors from /root/.cache/huggingface/hub/models--NousResearch--Nous-Hermes-2-Mistral-7B-DPO-GGUF/snapshots/eb85cf06e8663157611e8ee472e61b43f50ee49f/Nous-Hermes-2-Mistral-7B-DPO.Q4_K_M.gguf (version GGUF V3 (latest))
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = llama
llama_model_loader: - kv   1:                               general.name str              = models
llama_model_loader: - kv   2:                       llama.context_length u32              = 32768
llama_model_loader: - kv   3:                     llama.embedding_length u32              = 4096
llama_model_loader: - kv   4:                          llama.block_count u32              = 32
llama_model_loader: - kv   5:                  llama.feed_forward_length u32              = 14336
llama_model_loader: - kv   6:  

LlamaIndex LLM configured to use local LlamaCPP model.


CPU : SSE3 = 1 | SSSE3 = 1 | AVX = 1 | AVX2 = 1 | F16C = 1 | FMA = 1 | BMI2 = 1 | LLAMAFILE = 1 | OPENMP = 1 | REPACK = 1 | 
Model metadata: {'tokenizer.chat_template': "{% for message in messages %}{{'<|im_start|>' + message['role'] + '\n' + message['content'] + '<|im_end|>' + '\n'}}{% endfor %}{% if add_generation_prompt %}{{ '<|im_start|>assistant\n' }}{% endif %}", 'tokenizer.ggml.add_eos_token': 'false', 'tokenizer.ggml.eos_token_id': '32000', 'general.architecture': 'llama', 'llama.rope.freq_base': '10000.000000', 'llama.context_length': '32768', 'general.name': 'models', 'tokenizer.ggml.add_bos_token': 'true', 'llama.embedding_length': '4096', 'llama.feed_forward_length': '14336', 'llama.attention.layer_norm_rms_epsilon': '0.000010', 'llama.rope.dimension_count': '128', 'tokenizer.ggml.bos_token_id': '1', 'llama.attention.head_count': '32', 'llama.block_count': '32', 'llama.attention.head_count_kv': '8', 'general.quantization_version': '2', 'tokenizer.ggml.model': 'llama', 'gene

In [32]:
# Re-running the main installation cell with llama-index-llms-llama-cpp
!pip -q install -U \
  llama-index \
  llama-index-llms-openai \
  llama-index-embeddings-openai \
  llama-index-embeddings-huggingface \
  llama-index-llms-llama-cpp \
  llama-index-retrievers-bm25 \
  llama-index-readers-file \
  nest-asyncio

# Re-running the `llama-cpp-python` installation with --force-reinstall to ensure it's properly linked
!pip -q install llama-cpp-python --force-reinstall --no-deps

# Re-running the model download with the corrected filename
from huggingface_hub import hf_hub_download
import os

model_name = "NousResearch/Nous-Hermes-2-Mistral-7B-DPO-GGUF"
model_file = "Nous-Hermes-2-Mistral-7B-DPO.Q4_K_M.gguf" # Corrected filename

model_path = hf_hub_download(repo_id=model_name, filename=model_file)
print(f"Downloaded model to: {model_path}")

# Re-importing and configuring HuggingFaceEmbedding (if it was overridden by the previous errors)
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
import torch
if torch.cuda.is_available():
    device = "cuda"
else:
    device = "cpu"
Settings.embed_model = HuggingFaceEmbedding(
    model_name="BAAI/bge-small-en-v1.5",
    device=device
)

# Re-running the LLM configuration with LlamaCPP
from llama_index.llms.llama_cpp import LlamaCPP
from llama_index.core import Settings
from llama_index.core.node_parser import SentenceSplitter

llm = LlamaCPP(
    model_path=model_path,
    temperature=0.1,
    max_new_tokens=256,
    context_window=3900,
    generate_kwargs={},
    model_kwargs={"n_gpu_layers": -1}, # Uncomment this line to use GPU if available
    verbose=True,
)

Settings.llm = llm
Settings.node_parser = SentenceSplitter(chunk_size=256, chunk_overlap=40)

print("LlamaIndex LLM configured to use local LlamaCPP model.")

# Now, re-creating the vector store index and query engine with the local embedding and LLM
from llama_index.core import SimpleDirectoryReader, VectorStoreIndex
from llama_index.retrievers.bm25 import BM25Retriever
from llama_index.core.retrievers import QueryFusionRetriever
from llama_index.core.response_synthesizers import get_response_synthesizer
from llama_index.core.query_engine import RetrieverQueryEngine

documents = SimpleDirectoryReader("tech_kb").load_data()
vector_index = VectorStoreIndex.from_documents(documents)
vector_retriever = vector_index.as_retriever(similarity_top_k=4)

nodes = Settings.node_parser.get_nodes_from_documents(documents)
bm25_retriever = BM25Retriever.from_defaults(nodes=nodes, similarity_top_k=4)

fusion_retriever = QueryFusionRetriever(
    [vector_retriever, bm25_retriever],
    similarity_top_k=6,
    num_queries=4,
    mode="reciprocal_rerank",
    use_async=True,
    verbose=True,
)

response_synthesizer = get_response_synthesizer(response_mode="compact")

query_engine = RetrieverQueryEngine(
    retriever=fusion_retriever,
    response_synthesizer=response_synthesizer,
)

# Define the query (re-using the previous RAG concepts query)
query = """
Explain how query decomposition, hybrid retrieval, and reciprocal rank fusion
work together in an advanced RAG pipeline. Also mention why citations matter.
"""

print("Attempting to query with local LLM...")
response = query_engine.query(query)

print("\nFINAL ANSWER\n")
print(response)

print("\n" + "=" * 80)
print("SOURCE CITATIONS")
for i, source in enumerate(response.source_nodes, 1):
    print(f"\nSource {i}")
    print("File:", source.metadata.get("file_name"))
    print("Score:", source.score)
    print("Snippet:", source.text[:200])

print("Local LLM setup and RAG query complete.")

Downloaded model to: /root/.cache/huggingface/hub/models--NousResearch--Nous-Hermes-2-Mistral-7B-DPO-GGUF/snapshots/eb85cf06e8663157611e8ee472e61b43f50ee49f/Nous-Hermes-2-Mistral-7B-DPO.Q4_K_M.gguf


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

llama_model_loader: loaded meta data with 22 key-value pairs and 291 tensors from /root/.cache/huggingface/hub/models--NousResearch--Nous-Hermes-2-Mistral-7B-DPO-GGUF/snapshots/eb85cf06e8663157611e8ee472e61b43f50ee49f/Nous-Hermes-2-Mistral-7B-DPO.Q4_K_M.gguf (version GGUF V3 (latest))
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = llama
llama_model_loader: - kv   1:                               general.name str              = models
llama_model_loader: - kv   2:                       llama.context_length u32              = 32768
llama_model_loader: - kv   3:                     llama.embedding_length u32              = 4096
llama_model_loader: - kv   4:                          llama.block_count u32              = 32
llama_model_loader: - kv   5:                  llama.feed_forward_length u32              = 14336
llama_model_loader: - kv   6:  

LlamaIndex LLM configured to use local LlamaCPP model.


DEBUG:bm25s:Building index from IDs objects


Attempting to query with local LLM...


llama_perf_context_print:        load time =   32071.30 ms
llama_perf_context_print: prompt eval time =   32067.56 ms /    80 tokens (  400.84 ms per token,     2.49 tokens per second)
llama_perf_context_print:        eval time =   28637.69 ms /    40 runs   (  715.94 ms per token,     1.40 tokens per second)
llama_perf_context_print:       total time =   60724.93 ms /   120 tokens
llama_perf_context_print:    graphs reused =         39


Generated queries:
1. Query decomposition in advanced RAG pipeline
2. Hybrid retrieval and reciprocal rank fusion in advanced RAG pipeline
3. Importance of citations in advanced RAG pipeline


Llama.generate: 1 prefix-match hit, remaining 461 prompt tokens to eval
llama_perf_context_print:        load time =   32071.30 ms
llama_perf_context_print: prompt eval time =  151635.04 ms /   461 tokens (  328.93 ms per token,     3.04 tokens per second)
llama_perf_context_print:        eval time =  152638.03 ms /   208 runs   (  733.84 ms per token,     1.36 tokens per second)
llama_perf_context_print:       total time =  304393.35 ms /   669 tokens
llama_perf_context_print:    graphs reused =        206



FINAL ANSWER


In an advanced RAG pipeline, query decomposition, hybrid retrieval, and reciprocal rank fusion work together to improve the accuracy and recall of the system. 

Query decomposition breaks down a complex user question into smaller sub-queries. This improves recall because each sub-query can retrieve useful evidence. Fusion-based retrieval systems often combine query decomposition with multi-retriever search. 

Hybrid retrieval combines dense vector retrieval and sparse BM25 retrieval. Dense retrieval captures semantic similarity between the query and text, while BM25 captures keyword overlap and term importance. Reciprocal Rank Fusion combines ranked lists from multiple retrievers to improve retrieval quality. 

Citations matter because they improve trust, transparency, and verification in a RAG system. They show which source documents support the generated answer, allowing users to verify the accuracy of the response and providing a clear path to further explore the top

### Define a Second Query (Out of Context Example)
This cell defines a new query about a famous cricketer in India. This query is intentionally outside the scope of the provided `tech_kb` documents to demonstrate how the RAG pipeline handles irrelevant queries.

In [34]:
query = """
who is the famous cricketor in india
"""

### Display the Final Answer
This cell prints the final answer generated by the `query_engine` for the **previous** query (which was about cricketers, due to a possible re-run of a previous cell). The output clearly states that the information is not found in the context, demonstrating responsible AI behavior.

In [33]:
print("\nFINAL ANSWER\n")
print(response)


FINAL ANSWER


In an advanced RAG pipeline, query decomposition, hybrid retrieval, and reciprocal rank fusion work together to improve the accuracy and recall of the system. 

Query decomposition breaks down a complex user question into smaller sub-queries. This improves recall because each sub-query can retrieve useful evidence. Fusion-based retrieval systems often combine query decomposition with multi-retriever search. 

Hybrid retrieval combines dense vector retrieval and sparse BM25 retrieval. Dense retrieval captures semantic similarity between the query and text, while BM25 captures keyword overlap and term importance. Reciprocal Rank Fusion combines ranked lists from multiple retrievers to improve retrieval quality. 

Citations matter because they improve trust, transparency, and verification in a RAG system. They show which source documents support the generated answer, allowing users to verify the accuracy of the response and providing a clear path to further explore the top

### Display Source Citations
This cell displays the source documents (nodes) that the RAG pipeline used to generate the answer. For each source, it shows the filename, its relevance score, and a snippet of the text. This feature enhances transparency and allows users to verify the information. Note: The `response` object here refers to the response from the first query about RAG concepts, not the cricketer query, as indicated by the source snippets.

In [35]:
print("\n" + "=" * 80)
print("SOURCE CITATIONS")
for i, source in enumerate(response.source_nodes, 1):
    print(f"\nSource {i}")
    print("File:", source.metadata.get("file_name"))
    print("Score:", source.score)
    print("Snippet:", source.text[:200])


SOURCE CITATIONS

Source 1
File: rag_basics.txt
Score: 0.1317028027498678
Snippet: Retrieval-Augmented Generation, or RAG, combines retrieval with language generation.
A typical RAG pipeline retrieves relevant chunks and uses them as context for the LLM.
RAG reduces hallucination by

Source 2
File: query_decomposition.txt
Score: 0.1287930296391428
Snippet: Query decomposition breaks one complex user question into smaller sub-queries.
This improves recall because each sub-query can retrieve useful evidence.
Fusion-based retrieval systems often combine qu

Source 3
File: hybrid_search.txt
Score: 0.11454173067076291
Snippet: Hybrid retrieval combines dense vector retrieval and sparse BM25 retrieval.
Dense retrieval captures semantic similarity between query and text.
BM25 captures keyword overlap and term importance.
Reci

Source 4
File: reranking_note.txt
Score: 0.08014152250006296
Snippet: In production systems, a second-stage reranker may be used after retrieval.
In this lab, we use r